# Coffee17 exact checkpoint inventory V1
CPU-only. Attach available saved outputs from master and focused C1/C2 training (not the raw image dataset). This scans mounted checkpoint files and direct ZIP members against the 25 recorded best-checkpoint hashes. No training, no deletion, no deserialization. If no matches, check other saved versions; this does not prove permanent loss.

In [ ]:
import hashlib, json, os, zipfile
from pathlib import Path

EXPECTED = {
    "1": {
        "B0": "69ae135e0d3a14b9f6c78cd852596427928a01da323e7aa72bb81a3ec172edb8",
        "W3": "fb5a8f12b4bc31fc2d2c34219b150644777cc6b8d2a0aae1096348ca04b6fa14",
        "B1": "a9bb6e79c5f04da546a111a6fe2a1e8c11ab85de10af0265edf71f2c7f1b5094",
        "W4": "3d5d42ebf052d75043c6e9516e59675f83b0f47c5c205fa169382b9885c8a753",
        "C2": "d3ecc8d098572e3b4fb06da1cbe13f77125b8b9df0750254dcc673c20358e591"
    },
    "2": {
        "B0": "c760227caeec4209d8d6143903a505ebe8632e9015c842e84486f197b2c741cd",
        "W3": "43d581e4dc13032e5286de3437f68a0c635f8fd2af69970e49d49159f9a805ec",
        "B1": "aebea7da20a8625df50b5c96d0ec1520d5b4b4b297ebff6140ddf7e5d09d8fd1",
        "W4": "b3d8a98e99278d05cd2277b3502121cbfc287cef1091703ff57c8e94d3674124",
        "C2": "5bd510733711f20aa58e17d12f31911ef2c603c0745f9298033431cf0bd5f9bb"
    },
    "3": {
        "B0": "0f6703752ecbb21b90b23a55f8980a272ad35c0415966893885049b3b0ce87bd",
        "W3": "4ceded8dfd86d65fcc009ced49a90e0b521ebed251008add9d4fec6d7d033365",
        "B1": "13ec25b5142198e268666701bb3b6b7ca8b9efaadec8f9e4a83aa7fbb08dda24",
        "W4": "b17d3407f4a2e94d90e9dbce09b0e8b7221d0518a9604b43e5717267754e021e",
        "C2": "80683abab5c19012879049fd07dee3925ecd39900e8bc45a001cfa6a7d5f028f"
    },
    "4": {
        "B0": "79061a5901b657aaa9d65703009450496ab373a682319a59e85713023c162ca2",
        "W3": "c884a4d622652c10e11a6c2f66c0974110bce9bf327b0b0127ab6f2907afc2a8",
        "B1": "66f7eaabe40133c018fdfd1a4ef1dc9f2fc11954444b78ed00a62c2a721bf45e",
        "W4": "f41749cbf94341c6d7c2f76b74b7fedd2daa5175e4548b32db9b48e48cc43321",
        "C2": "7bf83ce940c4d25fd44f7a32be6096bda330ccd7f3fffbc9e54fb9e5b126b912"
    },
    "5": {
        "B0": "27fe57221233a5af33678b2104c7c44ada77a3ef6df08e67dc424f25593f75a4",
        "W3": "19c939c52c704bec7b49a9346a6dddff1c99e11f23db98329668cce023a7bb61",
        "B1": "09819f37253711092f978c407f1e4753e1e002146a4403151bf0d3521ea10f4c",
        "W4": "d9a8569b9d866e45fae06a179a5f20a22f1df3b82017d74e8411cf1bcac1511e",
        "C2": "b1a9e9c7506377c95cb972d37dada97c310c32565e8a105800c18b88490c9621"
    }
}
ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]
OUT = ROOTS[1] / "coffee17-checkpoint-inventory-v1.json"
assert ROOTS[1].is_dir(), "Run in Kaggle"
assert not OUT.exists(), "Existing report preserved; download or rename before repeating."
print("TRAINING: DISABLED")
print("CHECKPOINT LOAD: DISABLED (hash only)")
print("OUTER TEST: LOCKED")

lookup = {sha: (fold, arm) for fold, arms in EXPECTED.items() for arm, sha in arms.items()}
assert len(lookup) == 25
assert all(len(x) == 64 for x in lookup)
matches = {sha: [] for sha in lookup}
errors = []
checked = 0
archive_count = 0
weight_suffixes = {".pt", ".pth", ".ckpt", ".safetensors", ".bin"}

def digest(stream):
    h = hashlib.sha256()
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        h.update(chunk)
    return h.hexdigest()

assert digest(__import__("io").BytesIO(b"abc")) == "ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad"
def record(stream, location, size):
    global checked
    sha = digest(stream)
    checked += 1
    if sha in lookup:
        matches[sha].append({"location": location, "bytes": size})
        print("EXACT MATCH:", lookup[sha], location, flush=True)

def walk_error(exc):
    errors.append({"location": str(getattr(exc, "filename", "")), "error": str(exc)})

# Read only checkpoint-like files and archive directories. No torch import,
# checkpoint deserialization, extraction, image reading, or file deletion.
for root in ROOTS:
    if not root.is_dir():
        errors.append({"location": str(root), "error": "root unavailable"})
        continue
    for folder, dirs, files in os.walk(root, onerror=walk_error, followlinks=False):
        dirs.sort()
        for name in sorted(files):
            path = Path(folder) / name
            ext = path.suffix.lower()
            try:
                if ext in weight_suffixes:
                    with path.open("rb") as stream:
                        record(stream, str(path), path.stat().st_size)
                elif ext == ".zip":
                    archive_count += 1
                    with zipfile.ZipFile(path) as archive:
                        for member in archive.infolist():
                            if member.is_dir() or Path(member.filename).suffix.lower() not in weight_suffixes:
                                continue
                            try:
                                with archive.open(member) as stream:
                                    record(stream, f"{path}::{member.filename}", member.file_size)
                            except (OSError, RuntimeError, ValueError, zipfile.BadZipFile, NotImplementedError) as e:
                                errors.append({"location": f"{path}::{member.filename}", "error": str(e)})
            except (OSError, RuntimeError, ValueError, zipfile.BadZipFile, NotImplementedError) as e:
                errors.append({"location": str(path), "error": str(e)})

rows = [
    {"fold": int(fold), "arm": arm, "expected_sha256": sha,
     "status": "EXACT_HASH_FOUND" if matches[sha] else "NOT_FOUND_IN_SCANNED_SCOPE",
     "locations": matches[sha]}
    for fold, arms in EXPECTED.items() for arm, sha in arms.items()
]
counts = {arm: sum(r["arm"] == arm and bool(r["locations"]) for r in rows)
          for arm in ("B0", "C2", "W4", "W3", "B1")}
payload = {
    "format": "coffee17.checkpoint_inventory.v1",
    "master_package_sha256": "8d5c304c72ec8e1000f1a32e3ca0faa5dc6093466443f4c179068d40097272a3",
    "focused_package_sha256": "70365b4661d0a9b8509d68cd3ccfe58ac524dfbf54ed93f20512a44feca72a32",
    "scan_scope": [str(p) for p in ROOTS],
    "limitations": "Mounted files and direct ZIP members only; no nested ZIP, TAR, symlink directory traversal, unmounted Kaggle versions, or remote storage. Absence is not proof of permanent loss.",
    "training_performed": False,
    "checkpoint_deserialized": False,
    "outer_test_accessed": False,
    "weight_files_or_members_hashed": checked,
    "zip_archives_inspected": archive_count,
    "scan_errors": errors,
    "matched_counts": counts,
    "primary_exact_hashes_available": counts["B0"] == counts["C2"] == 5,
    "evaluation_authorized": False,
    "results": rows
}
with OUT.open("x", encoding="utf-8") as handle:
    json.dump(payload, handle, indent=2)
print("\nExact checkpoint matches by arm (out of 5):", counts)
print("Weight files/members hashed:", checked, "ZIP archives:", archive_count, "Errors:", len(errors))
print("READY:", OUT)
print("NO TRAINING. NO CHECKPOINT DELETION. NO TEST ACCESS.")

